# Diabetes Prediction

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [3]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split


In [4]:
## 1. Read and EDA 

train = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")

train.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [5]:
train.shape, test.shape

((700000, 26), (300000, 25))

In [6]:
train["diagnosed_diabetes"].value_counts(normalize = True)

diagnosed_diabetes
1.0    0.623296
0.0    0.376704
Name: proportion, dtype: float64

In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int64  
 11  heart_rate                

In [8]:
train.columns.tolist()

['id',
 'age',
 'alcohol_consumption_per_week',
 'physical_activity_minutes_per_week',
 'diet_score',
 'sleep_hours_per_day',
 'screen_time_hours_per_day',
 'bmi',
 'waist_to_hip_ratio',
 'systolic_bp',
 'diastolic_bp',
 'heart_rate',
 'cholesterol_total',
 'hdl_cholesterol',
 'ldl_cholesterol',
 'triglycerides',
 'gender',
 'ethnicity',
 'education_level',
 'income_level',
 'smoking_status',
 'employment_status',
 'family_history_diabetes',
 'hypertension_history',
 'cardiovascular_history',
 'diagnosed_diabetes']

In [9]:
train.isnull().sum().sort_values(ascending=False)

id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
smoking_status                        0
employment_status                     0
family_history_diabetes               0
hypertension_history                  0
cardiovascular_history                0


## X and Y (Big part) (Very beginning!)

In [10]:
# Find out X and Y
X = train.drop(columns=["diagnosed_diabetes", "id"])
y = train["diagnosed_diabetes"]

# Split!
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_test = test.drop(columns=["id"])

# Find categorical columns
cat_features = X.select_dtypes(include=["object"]).columns.tolist()


## 😁 PART 1: Baseline Model + Gradient Boosting (CatBoost)

In [11]:
# Start modeling
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)

0:	test: 0.6765062	best: 0.6765062 (0)	total: 655ms	remaining: 5m 26s
100:	test: 0.7041151	best: 0.7041151 (100)	total: 48.7s	remaining: 3m 12s
200:	test: 0.7097237	best: 0.7097237 (200)	total: 1m 36s	remaining: 2m 23s
300:	test: 0.7150133	best: 0.7150133 (300)	total: 2m 23s	remaining: 1m 34s
400:	test: 0.7189959	best: 0.7189959 (400)	total: 3m 10s	remaining: 47.1s
499:	test: 0.7209447	best: 0.7209447 (499)	total: 3m 57s	remaining: 0us

bestTest = 0.7209447064
bestIteration = 499



#### From this method, we can see the best score is 0.720944.

## 😍 PART 2: Refinement: Stratified K-Fold Cross Validation

In [12]:

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== Fold {fold} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",
        verbose=100
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    # validation prediction
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

    # test prediction（Avg. value）
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits



===== Fold 0 =====
0:	test: 0.6777144	best: 0.6777144 (0)	total: 548ms	remaining: 4m 33s
100:	test: 0.7056559	best: 0.7056559 (100)	total: 48.4s	remaining: 3m 11s
200:	test: 0.7109075	best: 0.7109075 (200)	total: 1m 35s	remaining: 2m 22s
300:	test: 0.7163116	best: 0.7163116 (300)	total: 2m 22s	remaining: 1m 34s
400:	test: 0.7199064	best: 0.7199064 (400)	total: 3m 9s	remaining: 46.8s
499:	test: 0.7216693	best: 0.7216693 (499)	total: 3m 56s	remaining: 0us

bestTest = 0.721669262
bestIteration = 499


===== Fold 1 =====
0:	test: 0.6752112	best: 0.6752112 (0)	total: 532ms	remaining: 4m 25s
100:	test: 0.7041586	best: 0.7041586 (100)	total: 48.4s	remaining: 3m 11s
200:	test: 0.7091740	best: 0.7091740 (200)	total: 1m 35s	remaining: 2m 22s
300:	test: 0.7141449	best: 0.7141449 (300)	total: 2m 23s	remaining: 1m 34s
400:	test: 0.7175194	best: 0.7175203 (398)	total: 3m 10s	remaining: 47.1s
499:	test: 0.7194006	best: 0.7194006 (499)	total: 3m 58s	remaining: 0us

bestTest = 0.7194006468
bestIterati

In [13]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y, oof_preds)
print("CV AUC:", auc)

CV AUC: 0.7209094759518814


#### From this method, we can find that the AUC score is 0.720909.


## 😎 PART 3: Ablation
After using 5-Kfold, lets investigate the future importance (Ablation) :

In [14]:
feature_importance = model.get_feature_importance()

fi_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance
}).sort_values(by="importance", ascending=False)

print(fi_df.head(15))

                               feature  importance
21             family_history_diabetes   36.502354
2   physical_activity_minutes_per_week   33.500913
0                                  age   12.969958
14                       triglycerides    5.989413
6                                  bmi    2.800812
13                     ldl_cholesterol    1.559128
3                           diet_score    1.053710
10                          heart_rate    0.937809
12                     hdl_cholesterol    0.893538
11                   cholesterol_total    0.870062
8                          systolic_bp    0.784700
7                   waist_to_hip_ratio    0.755809
5            screen_time_hours_per_day    0.430073
9                         diastolic_bp    0.244398
16                           ethnicity    0.119897


In [15]:
## We will drop the most impact features:

X_drop = X.drop(columns=[
    "family_history_diabetes",
    "physical_activity_minutes_per_week"
])

X_test_drop = X_test.drop(columns=[
    "family_history_diabetes",
    "physical_activity_minutes_per_week"
])



from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_drop))
test_preds = np.zeros(len(X_test_drop))

# 保险一点：更新一下 categorical columnsç
cat_features_drop = [col for col in cat_features if col in X_drop.columns]

for fold, (train_idx, val_idx) in enumerate(skf.split(X_drop, y)):
    print(f"\n===== Fold {fold} =====")

    X_train, X_val = X_drop.iloc[train_idx], X_drop.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",
        verbose=100
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features_drop
    )

    # validation prediction
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

    # test prediction（平均）
    test_preds += model.predict_proba(X_test_drop)[:, 1] / skf.n_splits

auc = roc_auc_score(y, oof_preds)
print("CV AUC after dropping 2 strong features:", auc)


===== Fold 0 =====
0:	test: 0.6000825	best: 0.6000825 (0)	total: 533ms	remaining: 4m 25s
100:	test: 0.6228410	best: 0.6228410 (100)	total: 48.6s	remaining: 3m 12s
200:	test: 0.6251812	best: 0.6251812 (200)	total: 1m 35s	remaining: 2m 22s
300:	test: 0.6270834	best: 0.6270834 (300)	total: 2m 21s	remaining: 1m 33s
400:	test: 0.6281054	best: 0.6281054 (400)	total: 3m 9s	remaining: 46.9s
499:	test: 0.6287441	best: 0.6287442 (498)	total: 3m 58s	remaining: 0us

bestTest = 0.6287441658
bestIteration = 498

Shrink model to first 499 iterations.

===== Fold 1 =====
0:	test: 0.6054916	best: 0.6054916 (0)	total: 516ms	remaining: 4m 17s
100:	test: 0.6251323	best: 0.6251323 (100)	total: 48s	remaining: 3m 9s
200:	test: 0.6273110	best: 0.6273110 (200)	total: 1m 34s	remaining: 2m 20s
300:	test: 0.6292724	best: 0.6292724 (300)	total: 2m 21s	remaining: 1m 33s
400:	test: 0.6305311	best: 0.6305325 (398)	total: 3m 9s	remaining: 46.8s
499:	test: 0.6310587	best: 0.6310587 (499)	total: 3m 57s	remaining: 0us



#### From this method, we can find that the AUC score is 0.630185.

## 🤪 PART 4: Feature Engineering! & Do 5Fold Part 2!

In [16]:
# interaction
X["age_bmi"] = X["age"] * X["bmi"]
X_test["age_bmi"] = X_test["age"] * X_test["bmi"]

# ratio (For medical use)
X["trig_hdl_ratio"] = X["triglycerides"] / (X["hdl_cholesterol"] + 1)
X_test["trig_hdl_ratio"] = X_test["triglycerides"] / (X_test["hdl_cholesterol"] + 1)

# lifestyle (Easier and Efficient)
X["lifestyle_score"] = X["diet_score"] + X["physical_activity_minutes_per_week"] - X["screen_time_hours_per_day"]
X_test["lifestyle_score"] = X_test["diet_score"] + X_test["physical_activity_minutes_per_week"] - X_test["screen_time_hours_per_day"]

In [17]:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# To be safe, update categorical columnsç again!
cat_features_updated = [col for col in cat_features if col in X.columns]

for fold, (train_idx, val_idx) in enumerate(skf2.split(X, y)):
    print(f"\n===== Fold {fold} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",
        verbose=100
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features_updated
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf2.n_splits

auc = roc_auc_score(y, oof_preds)
print("CV AUC with engineered features:", auc)


===== Fold 0 =====
0:	test: 0.6817666	best: 0.6817666 (0)	total: 557ms	remaining: 4m 38s
100:	test: 0.7058581	best: 0.7058581 (100)	total: 49.2s	remaining: 3m 14s
200:	test: 0.7108849	best: 0.7108849 (200)	total: 1m 38s	remaining: 2m 26s
300:	test: 0.7158377	best: 0.7158377 (300)	total: 2m 26s	remaining: 1m 36s
400:	test: 0.7194486	best: 0.7194486 (400)	total: 3m 15s	remaining: 48.2s
499:	test: 0.7212588	best: 0.7212588 (499)	total: 4m 3s	remaining: 0us

bestTest = 0.7212587773
bestIteration = 499


===== Fold 1 =====
0:	test: 0.6804863	best: 0.6804863 (0)	total: 544ms	remaining: 4m 31s
100:	test: 0.7032814	best: 0.7032814 (100)	total: 49.1s	remaining: 3m 14s
200:	test: 0.7090634	best: 0.7090634 (200)	total: 1m 37s	remaining: 2m 25s
300:	test: 0.7137520	best: 0.7137520 (300)	total: 2m 26s	remaining: 1m 36s
400:	test: 0.7174508	best: 0.7174508 (400)	total: 3m 14s	remaining: 48.1s
499:	test: 0.7195209	best: 0.7195228 (498)	total: 4m 3s	remaining: 0us

bestTest = 0.7195228148
bestIterati

#### From this method, we can find that the AUC score is 0.720286.

## 😄 Part 5: Final Submission

In [18]:
# 😁 Part 5: Final Submission (Best Model = Part 2 original 5-fold CatBoost)

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# 1. Rebuild clean original data
X_final = train.drop(columns=["diagnosed_diabetes", "id"])
y_final = train["diagnosed_diabetes"]
X_test_final = test.drop(columns=["id"])

# 2. Rebuild categorical feature list
cat_features_final = X_final.select_dtypes(include=["object"]).columns.tolist()

# 3. 5-fold CV using the same setup as Part 2
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_final = np.zeros(len(X_final))
test_preds_final = np.zeros(len(X_test_final))

for fold, (train_idx, val_idx) in enumerate(skf_final.split(X_final, y_final)):
    print(f"\n===== Final Fold {fold} =====")

    X_train_final, X_val_final = X_final.iloc[train_idx], X_final.iloc[val_idx]
    y_train_final, y_val_final = y_final.iloc[train_idx], y_final.iloc[val_idx]

    final_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",
        verbose=100
    )

    final_model.fit(
        X_train_final, y_train_final,
        eval_set=(X_val_final, y_val_final),
        cat_features=cat_features_final
    )

    oof_preds_final[val_idx] = final_model.predict_proba(X_val_final)[:, 1]
    test_preds_final += final_model.predict_proba(X_test_final)[:, 1] / skf_final.n_splits

final_auc = roc_auc_score(y_final, oof_preds_final)
print("Final CV AUC:", final_auc)

submission = pd.DataFrame({
    "id": test["id"],
    "diagnosed_diabetes": test_preds_final
})

submission.to_csv("submission.csv", index=False)
print("submission.csv has been saved!")
submission.head()


===== Final Fold 0 =====
0:	test: 0.6777144	best: 0.6777144 (0)	total: 528ms	remaining: 4m 23s
100:	test: 0.7056559	best: 0.7056559 (100)	total: 48.9s	remaining: 3m 13s
200:	test: 0.7109075	best: 0.7109075 (200)	total: 1m 37s	remaining: 2m 24s
300:	test: 0.7163116	best: 0.7163116 (300)	total: 2m 24s	remaining: 1m 35s
400:	test: 0.7199064	best: 0.7199064 (400)	total: 3m 11s	remaining: 47.3s
499:	test: 0.7216693	best: 0.7216693 (499)	total: 3m 58s	remaining: 0us

bestTest = 0.721669262
bestIteration = 499


===== Final Fold 1 =====
0:	test: 0.6752112	best: 0.6752112 (0)	total: 522ms	remaining: 4m 20s
100:	test: 0.7041586	best: 0.7041586 (100)	total: 48.5s	remaining: 3m 11s
200:	test: 0.7091740	best: 0.7091740 (200)	total: 1m 36s	remaining: 2m 23s
300:	test: 0.7141449	best: 0.7141449 (300)	total: 2m 23s	remaining: 1m 34s
400:	test: 0.7175194	best: 0.7175203 (398)	total: 3m 11s	remaining: 47.2s
499:	test: 0.7194006	best: 0.7194006 (499)	total: 3m 58s	remaining: 0us

bestTest = 0.719400646

,id,diagnosed_diabetes
0,700000,0.522660
1,700001,0.662281
2,700002,0.761527
3,700003,0.462338
4,700004,0.900482
